# 🏗️ Silver Layer Transformation Pipeline

This notebook transforms raw data from the Bronze layer into clean, enriched data in the Silver layer.

**What this pipeline does:**
- Reads trip data from Bronze tables (Yellow, Green, FHV, FHVHV)
- Standardizes column names and data types
- Enriches data with zone information and derived metrics
- Applies data quality rules and flags suspicious records
- Loads only new data incrementally (smart processing)
- Tracks processing metadata to avoid reprocessing



## 📦 Configuration & Setup

Let's start by defining our configuration. This is where you'll add new months of data in the future.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Database configuration
DATABASE = "nyc_taxi_lakehouse"
PIPELINE_VERSION = "1.0"

# Trip type configurations - this is where we define what to process
TRIP_CONFIGS = [
    {
        "trip_type": "yellow",
        "bronze_tables": [
            f"{DATABASE}.bronze.yellow_tripdata_2024_10",
            f"{DATABASE}.bronze.yellow_tripdata_2024_11"
            # 👆 Add new months here as they come in
        ],
        "silver_table": f"{DATABASE}.silver.yellow_trips"
    },
    {
        "trip_type": "green",
        "bronze_tables": [
            f"{DATABASE}.bronze.green_tripdata_2024_10",
            f"{DATABASE}.bronze.green_tripdata_2024_11"
            # 👆 Add new months here as they come in
        ],
        "silver_table": f"{DATABASE}.silver.green_trips"
    },
    {
        "trip_type": "fhv",
        "bronze_tables": [
            f"{DATABASE}.bronze.fhv_tripdata_2024_10",
            f"{DATABASE}.bronze.fhv_tripdata_2024_11"
            # 👆 Add new months here as they come in
        ],
        "silver_table": f"{DATABASE}.silver.fhv_trips"
    },
    {
        "trip_type": "fhvhv",
        "bronze_tables": [
            f"{DATABASE}.bronze.fhvhv_tripdata_2024_10",
            f"{DATABASE}.bronze.fhvhv_tripdata_2024_11"
            # 👆 Add new months here as they come in
        ],
        "silver_table": f"{DATABASE}.silver.fhvhv_trips"
    }
]

# Zone lookup table
ZONE_LOOKUP_TABLE = f"{DATABASE}.bronze.taxi_zone_lookup"
METADATA_TABLE = f"{DATABASE}.silver.processing_metadata"

print("✅ Configuration loaded successfully!")
print(f"📊 Processing {len(TRIP_CONFIGS)} trip types")

## 🎯 Helper Functions

These are our reusable building blocks that make the transformation logic clean and maintainable.

In [0]:
def get_already_processed():
    """
    Check what we've already processed to avoid duplicate work.
    Returns a set of (trip_type, year, month) tuples.
    """
    try:
        processed_df = spark.table(METADATA_TABLE).filter("status = 'SUCCESS'")
        processed = {(row.trip_type, row.year, row.month) for row in processed_df.collect()}
        print(f"📋 Found {len(processed)} already processed partitions")
        return processed
    except Exception as e:
        print(f"⚠️  Metadata table doesn't exist yet or is empty. Starting fresh!")
        return set()


def extract_year_month_from_table_name(table_name):
    """
    Extract year and month from table names like 'yellow_tripdata_2024_10'
    Returns: (year, month) as integers
    """
    parts = table_name.split("_")
    year = int(parts[-2])
    month = int(parts[-1])
    return year, month


def add_temporal_features(df, pickup_col="pickup_datetime"):
    """
    Add all our time-based derived columns in one go.
    This makes our data much more queryable for analytics!
    """
    df = df.withColumn("pickup_date", F.to_date(F.col(pickup_col)))
    df = df.withColumn("year", F.year(F.col(pickup_col)))
    df = df.withColumn("month", F.month(F.col(pickup_col)))
    df = df.withColumn("pickup_day", F.dayofmonth(F.col(pickup_col)))
    df = df.withColumn("pickup_hour", F.hour(F.col(pickup_col)))
    df = df.withColumn("pickup_day_of_week", F.dayofweek(F.col(pickup_col)))
    df = df.withColumn("pickup_day_name", F.date_format(F.col(pickup_col), 'EEEE'))
    df = df.withColumn("pickup_is_weekend", F.col("pickup_day_of_week").isin([1, 7]))
    
    # Time of day buckets
    df = df.withColumn("pickup_time_of_day", 
        F.when((F.col("pickup_hour") >= 0) & (F.col("pickup_hour") <= 5), "Night")
         .when((F.col("pickup_hour") >= 6) & (F.col("pickup_hour") <= 11), "Morning")
         .when((F.col("pickup_hour") >= 12) & (F.col("pickup_hour") <= 17), "Afternoon")
         .when((F.col("pickup_hour") >= 18) & (F.col("pickup_hour") <= 23), "Evening")
    )
    
    return df


def add_trip_metrics(df, trip_type):
    """
    Calculate trip duration, speed, and categorize distances.
    Different trip types have different columns available.
    """
    # Trip duration (all trip types have pickup and dropoff)
    df = df.withColumn("trip_duration_seconds", 
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime"))
    df = df.withColumn("trip_duration_minutes", F.col("trip_duration_seconds") / 60)
    
    # Distance categories and speed (only for types with distance data)
    if trip_type in ["yellow", "green", "fhvhv"]:
        df = df.withColumn("distance_category",
            F.when(F.col("trip_distance_miles") == 0, "Zero distance")
             .when(F.col("trip_distance_miles") <= 1, "0-1 miles")
             .when(F.col("trip_distance_miles") <= 3, "1-3 miles")
             .when(F.col("trip_distance_miles") <= 5, "3-5 miles")
             .when(F.col("trip_distance_miles") <= 10, "5-10 miles")
             .when(F.col("trip_distance_miles") > 10, "10+ miles")
             .otherwise("Unknown")
        )
        
        # Average speed calculation
        df = df.withColumn("average_speed_mph",
            F.when((F.col("trip_duration_minutes") > 0) & (F.col("trip_distance_miles") > 0),
                F.col("trip_distance_miles") / (F.col("trip_duration_seconds") / 3600))
             .otherwise(None)
        )
    
    # Passenger categories (yellow and green only)
    if trip_type in ["yellow", "green"]:
        df = df.withColumn("passenger_category",
            F.when(F.col("passenger_count") == 0, "Zero passengers")
             .when(F.col("passenger_count") == 1, "Solo")
             .when(F.col("passenger_count") == 2, "2 passengers")
             .when(F.col("passenger_count").isin([3, 4]), "3-4 passengers")
             .when(F.col("passenger_count") >= 5, "5+ passengers")
             .otherwise("Unknown")
        )
    
    return df


def enrich_with_zones(df, zone_lookup_df):
    """
    Join with zone lookup to add borough and zone names.
    We do this twice - once for pickup, once for dropoff.
    """
    # Pickup zone enrichment
    df = df.join(
        zone_lookup_df.select(
            F.col("LocationID").alias("pickup_loc_id"),
            F.col("Borough").alias("pickup_borough"),
            F.col("Zone").alias("pickup_zone"),
            F.col("service_zone").alias("pickup_service_zone")
        ),
        F.col("pickup_location_id") == F.col("pickup_loc_id"),
        "left"
    ).drop("pickup_loc_id")
    
    # Dropoff zone enrichment
    df = df.join(
        zone_lookup_df.select(
            F.col("LocationID").alias("dropoff_loc_id"),
            F.col("Borough").alias("dropoff_borough"),
            F.col("Zone").alias("dropoff_zone"),
            F.col("service_zone").alias("dropoff_service_zone")
        ),
        F.col("dropoff_location_id") == F.col("dropoff_loc_id"),
        "left"
    ).drop("dropoff_loc_id")
    
    return df


def apply_data_quality_rules(df, trip_type):
    """
    Flag suspicious records and filter out invalid ones.
    We flag issues but keep the data for analysis, except for truly invalid records.
    """
    # Build quality flags as an array
    quality_flags = []
    
    # Zero distance
    if trip_type in ["yellow", "green", "fhvhv"]:
        quality_flags.append(
            F.when(F.col("trip_distance_miles") == 0, "zero_distance")
        )
    
    # Passenger issues (yellow and green only)
    if trip_type in ["yellow", "green"]:
        quality_flags.append(
            F.when(F.col("passenger_count") == 0, "zero_passengers")
        )
        quality_flags.append(
            F.when(F.col("passenger_count").isNull(), "missing_passengers")
        )
    
    # Speed and duration issues (types with distance)
    if trip_type in ["yellow", "green", "fhvhv"]:
        quality_flags.append(
            F.when(F.col("average_speed_mph") > 80, "excessive_speed")
        )
        quality_flags.append(
            F.when((F.col("trip_duration_seconds") < 60) & (F.col("trip_distance_miles") > 1), "impossibly_fast")
        )
    
    # Long trips
    quality_flags.append(
        F.when(F.col("trip_duration_minutes") > 180, "excessive_duration")
    )
    
    # Location issues
    quality_flags.append(
        F.when(F.col("pickup_location_id").isNull(), "missing_pickup_location")
    )
    
    # Financial issues (yellow and green)
    if trip_type in ["yellow", "green"]:
        quality_flags.append(
            F.when(
                (F.col("congestion_surcharge") < 0) | 
                (F.col("improvement_surcharge") < 0) |
                (F.col("extra_charges") < 0),
                "negative_fees"
            )
        )
    
    # Combine all flags into array, filtering out nulls
    df = df.withColumn("data_quality_flags", 
        F.array_distinct(F.array_remove(F.array(*quality_flags), None))
    )
    
    # Mark as suspicious if any flags exist
    df = df.withColumn("is_suspicious", F.size("data_quality_flags") > 0)
    
    # Now filter out truly invalid records (removal rules)
    filter_conditions = [
        # Time sanity
        F.col("trip_duration_seconds") >= 0,
        F.col("pickup_datetime") <= F.current_timestamp(),
        # Location validity (1-265 is valid zone range)
        (F.col("pickup_location_id").between(1, 265)) | F.col("pickup_location_id").isNull(),
        (F.col("dropoff_location_id").between(1, 265)) | F.col("dropoff_location_id").isNull()
    ]
    
    # Financial and distance validity (only for certain trip types)
    if trip_type in ["yellow", "green"]:
        filter_conditions.extend([
            F.col("total_fare") >= 0,
            F.col("fare_amount") >= 0
        ])
    
    if trip_type in ["yellow", "green", "fhvhv"]:
        filter_conditions.append(
            (F.col("trip_distance_miles") <= 500) | F.col("trip_distance_miles").isNull()
        )
    
    # Apply all filter conditions
    from functools import reduce
    valid_df = df.filter(reduce(lambda a, b: a & b, filter_conditions))
    
    return valid_df


def add_metadata_columns(df):
    """
    Add processing metadata to track when data was loaded.
    """
    df = df.withColumn("processed_at", F.current_timestamp())
    # Assuming ingested_at exists in bronze, otherwise create it
    if "ingested_at" not in df.columns:
        df = df.withColumn("ingested_at", F.current_timestamp())
    
    return df

print("✅ Helper functions loaded!")

## 🗺️ Transformation Logic by Trip Type

Each trip type has its own transformation logic. Let's define them cleanly.

In [0]:
def transform_yellow_trips(bronze_df, zone_df):
    """
    Transform Yellow taxi data from Bronze to Silver.
    Yellow taxis are the iconic NYC yellow cabs.
    """
    # Column mapping and renaming
    df = bronze_df.select(
        F.col("VendorID").alias("vendor_id"),
        F.col("tpep_pickup_datetime").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("passenger_count"),
        F.col("trip_distance").alias("trip_distance_miles"),
        F.col("RatecodeID").alias("rate_code_id"),
        F.col("store_and_fwd_flag"),
        F.col("PULocationID").alias("pickup_location_id"),
        F.col("DOLocationID").alias("dropoff_location_id"),
        F.col("payment_type").alias("payment_type_id"),
        F.col("fare_amount"),
        F.col("extra").alias("extra_charges"),
        F.col("mta_tax"),
        F.col("tip_amount"),
        F.col("tolls_amount"),
        F.col("improvement_surcharge"),
        F.col("total_amount").alias("total_fare"),
        F.col("congestion_surcharge"),
        F.col("Airport_fee").alias("airport_fee")
    )
    
    # Decode vendor names
    df = df.withColumn("vendor_name",
        F.when(F.col("vendor_id") == 1, "Creative Mobile Technologies, LLC")
         .when(F.col("vendor_id") == 2, "Curb Mobility, LLC")
         .when(F.col("vendor_id") == 6, "Myle Technologies Inc")
         .otherwise("Unknown")
    )
    
    # Decode payment types
    df = df.withColumn("payment_method",
        F.when(F.col("payment_type_id") == 0, "Flex Fare")
         .when(F.col("payment_type_id") == 1, "Credit Card")
         .when(F.col("payment_type_id") == 2, "Cash")
         .when(F.col("payment_type_id") == 3, "No Charge")
         .when(F.col("payment_type_id") == 4, "Dispute")
         .otherwise("Unknown")
    )
    
    # Decode rate types
    df = df.withColumn("rate_type",
        F.when(F.col("rate_code_id") == 1, "Standard Rate")
         .when(F.col("rate_code_id") == 2, "JFK")
         .when(F.col("rate_code_id") == 3, "Newark")
         .when(F.col("rate_code_id") == 4, "Nassau or Westchester")
         .when(F.col("rate_code_id") == 5, "Negotiated Fare")
         .when(F.col("rate_code_id") == 6, "Group Ride")
         .when(F.col("rate_code_id") == 99, "Null/Unknown")
         .otherwise("Unknown")
    )
    
    # Convert Y/N to boolean
    df = df.withColumn("store_and_forward_flag",
        F.when(F.col("store_and_fwd_flag") == "Y", True)
         .when(F.col("store_and_fwd_flag") == "N", False)
         .otherwise(None)
    ).drop("store_and_fwd_flag")
    
    # Calculate tip percentage (only for credit card payments)
    df = df.withColumn("tip_percentage",
        F.when((F.col("payment_method") == "Credit Card") & (F.col("fare_amount") > 0),
            (F.col("tip_amount") / F.col("fare_amount")) * 100)
         .otherwise(None)
    )
    
    # Add temporal features
    df = add_temporal_features(df, "pickup_datetime")
    
    # Add trip metrics
    df = add_trip_metrics(df, "yellow")
    
    # Enrich with zones
    df = enrich_with_zones(df, zone_df)
    
    # Apply data quality rules
    df = apply_data_quality_rules(df, "yellow")
    
    # Add metadata
    df = add_metadata_columns(df)
    
    return df


def transform_green_trips(bronze_df, zone_df):
    """
    Transform Green taxi data from Bronze to Silver.
    Green taxis serve areas outside Manhattan's core.
    """
    # Column mapping and renaming
    df = bronze_df.select(
        F.col("VendorID").alias("vendor_id"),
        F.col("lpep_pickup_datetime").alias("pickup_datetime"),
        F.col("lpep_dropoff_datetime").alias("dropoff_datetime"),
        F.col("store_and_fwd_flag"),
        F.col("RatecodeID").alias("rate_code_id"),
        F.col("PULocationID").alias("pickup_location_id"),
        F.col("DOLocationID").alias("dropoff_location_id"),
        F.col("passenger_count"),
        F.col("trip_distance").alias("trip_distance_miles"),
        F.col("fare_amount"),
        F.col("extra").alias("extra_charges"),
        F.col("mta_tax"),
        F.col("tip_amount"),
        F.col("tolls_amount"),
        F.col("improvement_surcharge"),
        F.col("total_amount").alias("total_fare"),
        F.col("payment_type").alias("payment_type_id"),
        F.col("trip_type").alias("trip_type_id"),
        F.col("congestion_surcharge")
        # Note: ehail_fee is dropped as per spec
    )
    
    # Decode vendor names
    df = df.withColumn("vendor_name",
        F.when(F.col("vendor_id") == 1, "Creative Mobile Technologies, LLC")
         .when(F.col("vendor_id") == 2, "Curb Mobility, LLC")
         .otherwise("Unknown")
    )
    
    # Decode payment types
    df = df.withColumn("payment_method",
        F.when(F.col("payment_type_id") == 1, "Credit Card")
         .when(F.col("payment_type_id") == 2, "Cash")
         .when(F.col("payment_type_id") == 3, "No Charge")
         .when(F.col("payment_type_id") == 4, "Dispute")
         .when(F.col("payment_type_id") == 5, "Unknown")
         .otherwise("Unknown")
    )
    
    # Decode rate types
    df = df.withColumn("rate_type",
        F.when(F.col("rate_code_id") == 1, "Standard Rate")
         .when(F.col("rate_code_id") == 2, "JFK")
         .when(F.col("rate_code_id") == 3, "Newark")
         .when(F.col("rate_code_id") == 4, "Nassau or Westchester")
         .when(F.col("rate_code_id") == 5, "Negotiated Fare")
         .when(F.col("rate_code_id") == 6, "Group Ride")
         .when(F.col("rate_code_id") == 99, "Null/Unknown")
         .otherwise("Unknown")
    )
    
    # Decode trip type (dispatch vs street-hail)
    df = df.withColumn("dispatch_type",
        F.when(F.col("trip_type_id") == 1, "Street-hail")
         .when(F.col("trip_type_id") == 2, "Dispatch")
         .otherwise("Unknown")
    )
    
    # Convert Y/N to boolean
    df = df.withColumn("store_and_forward_flag",
        F.when(F.col("store_and_fwd_flag") == "Y", True)
         .when(F.col("store_and_fwd_flag") == "N", False)
         .otherwise(None)
    ).drop("store_and_fwd_flag")
    
    # Calculate tip percentage
    df = df.withColumn("tip_percentage",
        F.when((F.col("payment_method") == "Credit Card") & (F.col("fare_amount") > 0),
            (F.col("tip_amount") / F.col("fare_amount")) * 100)
         .otherwise(None)
    )
    
    # Add temporal features
    df = add_temporal_features(df, "pickup_datetime")
    
    # Add trip metrics
    df = add_trip_metrics(df, "green")
    
    # Enrich with zones
    df = enrich_with_zones(df, zone_df)
    
    # Apply data quality rules
    df = apply_data_quality_rules(df, "green")
    
    # Add metadata
    df = add_metadata_columns(df)
    
    return df


def transform_fhv_trips(bronze_df, zone_df):
    """
    Transform FHV (For-Hire Vehicle) data from Bronze to Silver.
    FHV includes services like Uber, Lyft, and traditional livery cabs.
    """
    # Column mapping and renaming
    df = bronze_df.select(
        F.col("dispatching_base_num").alias("dispatching_base_number"),
        F.col("pickup_datetime"),
        F.col("dropOff_datetime").alias("dropoff_datetime"),
        F.col("PUlocationID").alias("pickup_location_id"),
        F.col("DOlocationID").alias("dropoff_location_id"),
        F.col("Affiliated_base_number").alias("affiliated_base_number")
        # Note: SR_Flag is dropped as per spec
    )
    
    # Add temporal features
    df = add_temporal_features(df, "pickup_datetime")
    
    # Add trip metrics (FHV doesn't have distance or financial data)
    df = add_trip_metrics(df, "fhv")
    
    # Enrich with zones
    df = enrich_with_zones(df, zone_df)
    
    # Apply data quality rules
    df = apply_data_quality_rules(df, "fhv")
    
    # Add metadata
    df = add_metadata_columns(df)
    
    # Deduplicate FHV records
    df = df.dropDuplicates(["dispatching_base_number", "pickup_datetime", "dropoff_datetime"])
    
    return df


def transform_fhvhv_trips(bronze_df, zone_df):
    """
    Transform FHVHV (High-Volume For-Hire Vehicle) data from Bronze to Silver.
    This is specifically Uber and Lyft trips with detailed information.
    """
    # Column mapping and renaming
    df = bronze_df.select(
        F.col("hvfhs_license_num").alias("hvfhs_license_number"),
        F.col("dispatching_base_num").alias("dispatching_base_number"),
        F.col("originating_base_num").alias("originating_base_number"),
        F.col("request_datetime"),
        F.col("on_scene_datetime"),
        F.col("pickup_datetime"),
        F.col("dropoff_datetime"),
        F.col("PULocationID").alias("pickup_location_id"),
        F.col("DOLocationID").alias("dropoff_location_id"),
        F.col("trip_miles").alias("trip_distance_miles"),
        F.col("trip_time").alias("trip_time_seconds"),
        F.col("base_passenger_fare"),
        F.col("tolls").alias("tolls_amount"),
        F.col("bcf").alias("black_car_fund"),
        F.col("sales_tax"),
        F.col("congestion_surcharge"),
        F.col("airport_fee"),
        F.col("tips").alias("tip_amount"),
        F.col("driver_pay"),
        F.col("shared_request_flag"),
        F.col("shared_match_flag"),
        F.col("access_a_ride_flag"),
        F.col("wav_request_flag"),
        F.col("wav_match_flag")
    )
    
    # Decode company names
    df = df.withColumn("company_name",
        F.when(F.col("hvfhs_license_number") == "HV0003", "Uber")
         .when(F.col("hvfhs_license_number") == "HV0005", "Lyft")
         .otherwise("Unknown")
    )
    
    # Convert Y/N flags to boolean
    for flag_col in ["shared_request_flag", "shared_match_flag", "access_a_ride_flag", 
                     "wav_request_flag", "wav_match_flag"]:
        df = df.withColumn(flag_col,
            F.when(F.col(flag_col) == "Y", True)
             .when(F.col(flag_col) == "N", False)
             .otherwise(None)
        )
    
    # Calculate tip percentage
    df = df.withColumn("tip_percentage",
        F.when(F.col("base_passenger_fare") > 0,
            (F.col("tip_amount") / F.col("base_passenger_fare")) * 100)
         .otherwise(None)
    )
    
    # Add temporal features
    df = add_temporal_features(df, "pickup_datetime")
    
    # Add trip metrics
    df = add_trip_metrics(df, "fhvhv")
    
    # Enrich with zones
    df = enrich_with_zones(df, zone_df)
    
    # Apply data quality rules
    df = apply_data_quality_rules(df, "fhvhv")
    
    # Add metadata
    df = add_metadata_columns(df)
    
    return df

print("✅ Transformation functions loaded!")

## 🚀 Main Processing Pipeline

This is where the magic happens! We loop through each trip type and process only what's new.

In [0]:
# Load zone lookup once (it's used for all trip types)
zone_lookup_df = spark.table(ZONE_LOOKUP_TABLE)
print(f"📍 Loaded {zone_lookup_df.count()} zone records")

# Get what we've already processed
already_processed = get_already_processed()

# Map trip types to their transformation functions
TRANSFORM_FUNCTIONS = {
    "yellow": transform_yellow_trips,
    "green": transform_green_trips,
    "fhv": transform_fhv_trips,
    "fhvhv": transform_fhvhv_trips
}

# Process each trip type
for config in TRIP_CONFIGS:
    trip_type = config["trip_type"]
    bronze_tables = config["bronze_tables"]
    silver_table = config["silver_table"]
    
    print(f"\n{'='*80}")
    print(f"🚕 Processing {trip_type.upper()} trips")
    print(f"{'='*80}")
    
    # Get transformation function
    transform_func = TRANSFORM_FUNCTIONS[trip_type]
    
    # Process each bronze table for this trip type
    for bronze_table in bronze_tables:
        # Extract year and month from table name
        year, month = extract_year_month_from_table_name(bronze_table.split(".")[-1])
        
        # Check if we've already processed this
        if (trip_type, year, month) in already_processed:
            print(f"⏭️  Skipping {bronze_table} - already processed")
            continue
        
        print(f"\n📥 Reading from: {bronze_table}")
        print(f"   Year: {year}, Month: {month}")
        
        try:
            # Read bronze data
            bronze_df = spark.table(bronze_table)
            row_count_bronze = bronze_df.count()
            print(f"   📊 Bronze records: {row_count_bronze:,}")
            
            # Transform the data
            print(f"   🔄 Transforming...")
            silver_df = transform_func(bronze_df, zone_lookup_df)
            row_count_silver = silver_df.count()
            print(f"   ✨ Silver records: {row_count_silver:,}")
            print(f"   🗑️  Filtered out: {row_count_bronze - row_count_silver:,} invalid records")
            
            # Write to silver table (append mode, partitioned by year and month)
            print(f"   💾 Writing to: {silver_table}")
            silver_df.write \
                .format("delta") \
                .mode("append") \
                .partitionBy("year", "month") \
                .saveAsTable(silver_table)
            
            # Record successful processing in metadata
            metadata_row = [(
                trip_type,
                year,
                month,
                datetime.now(),
                row_count_silver,
                "SUCCESS",
                None,
                PIPELINE_VERSION
            )]
            
            metadata_schema = StructType([
                StructField("trip_type", StringType(), False),
                StructField("year", IntegerType(), False),
                StructField("month", IntegerType(), False),
                StructField("processed_at", TimestampType(), False),
                StructField("row_count", LongType(), False),
                StructField("status", StringType(), False),
                StructField("error_message", StringType(), True),
                StructField("pipeline_version", StringType(), False)
            ])
            
            metadata_df = spark.createDataFrame(metadata_row, schema=metadata_schema)
            metadata_df.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
            
            print(f"   ✅ Successfully processed {trip_type} {year}-{month:02d}")
            
        except Exception as e:
            print(f"   ❌ Error processing {bronze_table}: {str(e)}")
            
            # Record failed processing in metadata
            metadata_row = [(
                trip_type,
                year,
                month,
                datetime.now(),
                0,
                "FAILED",
                str(e)[:500],  # Truncate long error messages
                PIPELINE_VERSION
            )]
            
            metadata_df = spark.createDataFrame(metadata_row, schema=metadata_schema)
            metadata_df.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
            
            # Continue with next table instead of failing entire pipeline
            continue

print(f"\n{'='*80}")
print("🎉 Pipeline completed!")
print(f"{'='*80}")

## 📊 Processing Summary

Let's see what we just processed!

In [0]:
# Show processing summary
summary_df = spark.table(METADATA_TABLE) \
    .orderBy(F.col("processed_at").desc()) \
    .limit(20)

display(summary_df)

## 🔍 Quick Data Quality Check

Let's spot-check our silver tables to make sure everything looks good.

In [0]:
# Check row counts per partition
for config in TRIP_CONFIGS:
    trip_type = config["trip_type"]
    silver_table = config["silver_table"]
    
    print(f"\n📊 {trip_type.upper()} trips by partition:")
    partition_counts = spark.table(silver_table) \
        .groupBy("year", "month") \
        .count() \
        .orderBy("year", "month")
    
    display(partition_counts)

## 📝 Quality Flags Summary

Let's see how many suspicious records we flagged and why.

In [0]:
for config in TRIP_CONFIGS[:2]:  # Just check yellow and green for now
    trip_type = config["trip_type"]
    silver_table = config["silver_table"]
    
    print(f"\n🚩 {trip_type.upper()} quality flags:")
    
    flags_df = spark.table(silver_table) \
        .filter("is_suspicious = true") \
        .select(F.explode("data_quality_flags").alias("flag")) \
        .groupBy("flag") \
        .count() \
        .orderBy(F.col("count").desc())
    
    display(flags_df)

---

## 🎯 Next Steps

**To add new months of data:**
1. Go to the Configuration cell at the top
2. Add new table names to the appropriate `bronze_tables` lists
3. Run the notebook - it will automatically process only the new data!

**The pipeline will:**
- ✅ Skip already processed partitions
- ✅ Process only new data
- ✅ Track everything in metadata
- ✅ Handle errors gracefully

**Monitoring:**
- Check the `processing_metadata` table to see what's been processed
- Look for records with `status = 'FAILED'` to catch any issues
- Review quality flags to understand data issues

---